# Inflow forecasting

Forecast the next 24 hours of **inflow** using past observations (lags/rolling
stats) plus **precipitation** as an exogenous variable. Same engine as
`main.py` / `main_afrr.py`: `MLForecast` with `LinearRegression`, `Ridge`,
`RandomForestRegressor`, `XGBRegressor`, `GaussianProcessRegressor`, and a
simple ensemble -- evaluated with a rolling-origin backtest.

> **Note on `GaussianProcessRegressor`:** `src/model.py` now exposes a
> `MODEL_REGISTRY` of all available models by name, plus `DEFAULT_MODELS`
> (the original four). `build_mlf()`/`backtest()` take an optional `models`
> list to select a subset -- any sklearn-API regressor (`fit`/`predict`)
> can be registered the same way, no extra glue code needed. `main.py` and
> `main_afrr.py` are untouched and keep using `DEFAULT_MODELS`. This
> notebook opts into `GaussianProcessRegressor` explicitly, since it's
> O(n^3) in training rows and only cheap here because the inflow series
> is short (~500 training rows) -- it would be markedly slower on the
> larger heat_demand/aFRR series.
>
> **Note on the data:** `data/inflow_and_weather_data.parquet` only covers
> **2023-02-01 -> 2023-03-01** (28 days, hourly). That's short for a 7-day
> lag feature (`168h`), so the backtest below only has room for 7 rolling
> 24h windows in the last week of data. Treat results as a proof of concept,
> not a production-grade evaluation -- more history would give a much more
> reliable read.
>
> **Note on a fix required in `loaders/`:** the raw timestamps are
> timezone-aware (UTC). `mlforecast` silently drops the timezone when it
> computes internal cutoff dates, which then crashes when comparing against
> the (still tz-aware) exogenous `X_df` at predict time. `loaders/inflow.py`
> works around this by dropping the tz (keeping UTC wall-clock time),
> matching the convention already used in `loaders/heat_demand.py`. Nothing
> else in `src/` needed to change.

In [1]:
from config import HORIZON, COLORS
from src.data import split, resample
from src.model import build_mlf, fit, predict
from src.backtest import backtest
from src.plot import plot_backtest
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse, mape, smape
import loaders.inflow as loader

import polars as pl
import plotly.express as px

## Data

In [2]:
DATA_PATH = "data/inflow_and_weather_data.parquet"
FREQ      = "1h"
EXOG      = ["precip_past1min"]  # rain as exogenous
MODELS    = ["LinearRegression", "Ridge", "RandomForestRegressor", "XGBRegressor", "GaussianProcessRegressor"]

# -- Data -----------------------------------------------------------------
df = resample(loader.load(DATA_PATH), FREQ)
train_df, test_df = split(
    df,
    train_start="2023-02-01",
    test_start="2023-02-22",
    test_end="2023-03-01",
)
df.tail()

Train: 2023-02-01 00:00:00 → 2023-02-21 23:00:00  (504 rows)
Test:  2023-02-22 00:00:00  → 2023-03-01 00:00:00   (169 rows)


ds,y,precip_past1min,wind_speed,humidity,unique_id
datetime[μs],f64,f64,f64,f64,str
2023-02-28 20:00:00,380.296498,0.0,197.013333,0.0,"""inflow"""
2023-02-28 21:00:00,370.980874,0.0,230.003333,0.0,"""inflow"""
2023-02-28 22:00:00,357.101328,0.0,216.975,0.0,"""inflow"""
2023-02-28 23:00:00,341.393248,0.0,267.005,0.0,"""inflow"""
2023-03-01 00:00:00,334.348476,0.0,4.4,0.0,"""inflow"""


## Fit & predict

In [3]:
mlf = build_mlf(freq=FREQ, models=MODELS)

fit(mlf, train_df, horizon=HORIZON, exog=EXOG)

forecast_df = predict(mlf, test_df, horizon=HORIZON, exog=EXOG)

forecast_df

/home/alqua/git/forecastx/.venv/lib/python3.12/site-packages/mlforecast/forecast.py:873: UserWarning: Please rerun the `fit` method passing a proper value to prediction intervals to compute them.
  warnings.warn(warn_msg, UserWarning)


unique_id,ds,LinearRegression,Ridge,RandomForestRegressor,XGBRegressor,GaussianProcessRegressor,y
str,datetime[μs],f64,f64,f64,f64,f64,f64
"""inflow""",2023-02-22 00:00:00,378.488002,377.842838,370.65429,378.506993,417.540919,379.773163
"""inflow""",2023-02-22 01:00:00,370.52099,368.770111,356.171727,375.179323,402.905993,368.871931
"""inflow""",2023-02-22 02:00:00,362.198404,359.537174,346.361013,367.870541,390.487389,361.323619
"""inflow""",2023-02-22 03:00:00,355.788081,352.65426,339.963305,369.348673,381.140549,353.978106
"""inflow""",2023-02-22 04:00:00,357.306494,354.045481,338.613551,378.538543,378.291746,350.926121
…,…,…,…,…,…,…,…
"""inflow""",2023-02-22 19:00:00,439.608978,440.575338,421.269056,463.629351,433.686062,413.713949
"""inflow""",2023-02-22 20:00:00,419.826063,421.773847,412.0224,455.446418,422.411503,408.741949
"""inflow""",2023-02-22 21:00:00,399.480641,401.875838,403.002568,444.417796,410.742207,396.799595


## Backtest

Rolling-origin backtest over the test window (24h horizon, 24h step, no refit).

In [4]:
predictions_df, windows_df = backtest(
    train_df=train_df,
    test_df=test_df,
    horizon=HORIZON,
    freq=FREQ,
    exog=EXOG,
    refit=True,
    models=MODELS,
    ensemble_weights={m: 1 for m in MODELS},
)
windows_df

Backtesting 7 windows | refit=True | exog=['precip_past1min']


window_id,train_start,train_end,train_rows,test_start,test_end
i64,datetime[μs],datetime[μs],i64,datetime[μs],datetime[μs]
0,2023-02-01 00:00:00,2023-02-21 23:00:00,504,2023-02-22 00:00:00,2023-02-22 23:00:00
1,2023-02-01 00:00:00,2023-02-22 23:00:00,528,2023-02-23 00:00:00,2023-02-23 23:00:00
2,2023-02-01 00:00:00,2023-02-23 23:00:00,552,2023-02-24 00:00:00,2023-02-24 23:00:00
3,2023-02-01 00:00:00,2023-02-24 23:00:00,576,2023-02-25 00:00:00,2023-02-25 23:00:00
4,2023-02-01 00:00:00,2023-02-25 23:00:00,600,2023-02-26 00:00:00,2023-02-26 23:00:00
5,2023-02-01 00:00:00,2023-02-26 23:00:00,624,2023-02-27 00:00:00,2023-02-27 23:00:00
6,2023-02-01 00:00:00,2023-02-27 23:00:00,648,2023-02-28 00:00:00,2023-02-28 23:00:00


In [5]:
plot_backtest(predictions_df)

## Metrics

MAE, RMSE, MAPE, sMAPE per model, computed per backtest window and then averaged.

In [6]:
ci_cols = [c for c in predictions_df.columns if "-lo-" in c or "-hi-" in c]

metrics_by_window = evaluate(
    predictions_df.drop(ci_cols),
    metrics=[mae, rmse, mape, smape],
    train_df=train_df,
)
metrics_by_window

unique_id,cutoff,metric,LinearRegression,Ridge,RandomForestRegressor,XGBRegressor,GaussianProcessRegressor,ensemble
str,datetime[μs],str,f64,f64,f64,f64,f64,f64
"""inflow""",2023-02-22 00:00:00,"""mae""",14.256196,13.517782,7.244077,39.733665,22.115699,15.447149
"""inflow""",2023-02-23 00:00:00,"""mae""",34.672605,32.064568,13.949814,42.855931,26.672438,29.46173
"""inflow""",2023-02-24 00:00:00,"""mae""",71.880162,68.595983,106.565614,78.650081,99.422743,37.071705
"""inflow""",2023-02-25 00:00:00,"""mae""",57.684316,60.090713,68.782599,66.972242,99.567177,57.689245
"""inflow""",2023-02-26 00:00:00,"""mae""",35.375429,35.361228,86.057497,82.023176,138.1329,63.942403
…,…,…,…,…,…,…,…,…
"""inflow""",2023-02-24 00:00:00,"""smape""",0.067399,0.065826,0.113588,0.080086,0.104473,0.034875
"""inflow""",2023-02-25 00:00:00,"""smape""",0.060867,0.063277,0.066514,0.064233,0.107679,0.057515
"""inflow""",2023-02-26 00:00:00,"""smape""",0.045164,0.044479,0.095774,0.090623,0.146222,0.072949


In [7]:
overall_metrics = (
    metrics_by_window
    .drop("cutoff", "unique_id")
    .group_by("metric", maintain_order=True)
    .mean()
    .sort("metric")
)
overall_metrics

metric,LinearRegression,Ridge,RandomForestRegressor,XGBRegressor,GaussianProcessRegressor,ensemble
str,f64,f64,f64,f64,f64,f64
"""mae""",37.432854,37.190332,48.549219,52.415586,60.152831,32.984008
"""mape""",0.086181,0.085906,0.105184,0.117154,0.134942,0.074135
"""rmse""",46.193517,45.223835,58.905714,63.654004,71.083387,42.565589
"""smape""",0.042026,0.04189,0.054258,0.058548,0.066668,0.036707


In [8]:
metrics_long = overall_metrics.unpivot(
    index="metric", variable_name="model", value_name="value"
)

fig = px.bar(
    metrics_long, x="model", y="value", facet_col="metric",
    facet_col_wrap=2, color="model",
    color_discrete_sequence=COLORS + ["slateblue", "gold"],
    title="Backtest metrics by model (lower is better)",
)
fig.update_yaxes(matches=None, showticklabels=True)
fig.update_layout(showlegend=False)
fig.show()